# 01 — Verificación de Entorno Reproducible, Matriz Analítica y Protocolo Ético
**Proyecto:** Youth Employment Bridge (YEB) — Módulo de Analítica de Empleo  
**Issue:** `#01 [Data/Setup]: Estructura reproducible para el módulo de analítica de empleo`  
**Criterios de Aceptación:** `CA-ISS-01-01`, `CA-ISS-01-02`, `CA-ISS-01-03`, `CA-ISS-01-04`  
**Riesgos mitigados:** `LEG-001`, `TEC-007` | **Casos de prueba:** `CP-041`, `CP-041B`, `CP-041C`

---
### Objetivo de este Notebook
1. Comprobar las versiones instaladas de las librerías base (`pandas`, `numpy`, `jupyter`, `faker`).
2. Validar la estructura de directorios (`data/raw/`, `data/processed/`, `notebooks/`, `scripts/`, `outputs/`) utilizando **exclusivamente rutas relativas** mediante `pathlib.Path`.
3. Formular y estructurar la **Matriz de Preguntas Analíticas Iniciales** sobre el dominio de empleabilidad juvenil de YEB.
4. Documentar y verificar programáticamente la **Declaración Ética de Minimización y Anonimización** de datos sensibles (con protección reforzada para candidatos menores de 16–17 años).

## 1. Verificación del Intérprete y Versiones de Librerías (`CA-ISS-01-01`, `CA-ISS-01-02`)

In [1]:
import hashlib
import importlib.metadata
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from faker import Faker

required_packages = ["pandas", "numpy", "jupyter_core", "notebook", "Faker"]
env_report = [
    {"Componente": "Python", "Version": sys.version.split()[0], "Estado": "OK"},
    {"Componente": "Sistema Operativo", "Version": f"{platform.system()} {platform.release()}", "Estado": "OK"},
]

for pkg in required_packages:
    ver = importlib.metadata.version(pkg)
    env_report.append({"Componente": pkg, "Version": ver, "Estado": "OK"})

df_env = pd.DataFrame(env_report)
display(df_env)

,Componente,Version,Estado
0,Python,3.14.6,OK
1,Sistema Operativo,Windows 11,OK
2,pandas,3.1.0,OK
3,numpy,2.5.4,OK
4,jupyter_core,5.9.1,OK
5,notebook,7.6.4,OK
6,Faker,40.44.0,OK


## 2. Validación de Rutas Relativas con `pathlib.Path` (`CA-ISS-01-02`)
Para mitigar el riesgo **`TEC-007`** (rompimiento de reproducibilidad entre máquinas por rutas absolutas fijas), la raíz del subentorno `analytics/` se resuelve dinámicamente de forma relativa al directorio de trabajo.

In [2]:
# Resolución dinámica y relativa de la raíz analytics/
cwd = Path.cwd()
if cwd.name == "notebooks":
    ANALYTICS_ROOT = cwd.parent
elif (cwd / "analytics").is_dir():
    ANALYTICS_ROOT = cwd / "analytics"
else:
    ANALYTICS_ROOT = cwd

expected_paths = {
    "data/raw": ANALYTICS_ROOT / "data" / "raw",
    "data/processed": ANALYTICS_ROOT / "data" / "processed",
    "notebooks": ANALYTICS_ROOT / "notebooks",
    "scripts": ANALYTICS_ROOT / "scripts",
    "outputs": ANALYTICS_ROOT / "outputs",
    "docs": ANALYTICS_ROOT / "docs",
    "requirements.txt": ANALYTICS_ROOT / "requirements.txt",
}

path_records = []
for label, path_obj in expected_paths.items():
    exists = path_obj.exists()
    kind = "Directorio" if path_obj.is_dir() else ("Archivo" if path_obj.is_file() else "Ausente")
    rel_path = path_obj.relative_to(ANALYTICS_ROOT).as_posix()
    assert exists, f"Ruta requerida no encontrada: {rel_path}"
    path_records.append({
        "Recurso": label,
        "Ruta_Relativa": f"analytics/{rel_path}",
        "Tipo": kind,
        "Existe": exists,
    })

df_paths = pd.DataFrame(path_records)
display(df_paths)

,Recurso,Ruta_Relativa,Tipo,Existe
0,data/raw,analytics/data/raw,Directorio,True
1,data/processed,analytics/data/processed,Directorio,True
2,notebooks,analytics/notebooks,Directorio,True
3,scripts,analytics/scripts,Directorio,True
4,outputs,analytics/outputs,Directorio,True
5,docs,analytics/docs,Directorio,True
6,requirements.txt,analytics/requirements.txt,Archivo,True


## 3. Matriz de Preguntas Analíticas Iniciales sobre el Dominio YEB (`CA-ISS-01-03`)
El modelo de datos transaccional de **Youth Employment Bridge** (`users`, `companies`, `jobs`, `applications`) permite responder cuatro preguntas clave sobre la inserción laboral juvenil:

In [3]:
analytical_matrix = [
    {
        "ID": "PA-01",
        "Eje_Analitico": "Embudo de selección",
        "Pregunta_Negocio": "¿Cuál es la tasa de conversión y descarte entre estados (pending -> review -> interview -> accepted/rejected)?",
        "Entidades_YEB": "applications.status, users.is_minor",
        "Indicador_Objetivo": "Tasa de conversión por etapa (%)",
    },
    {
        "ID": "PA-02",
        "Eje_Analitico": "Tiempo promedio de contratación",
        "Pregunta_Negocio": "¿Cuántos días transcurren en promedio desde la postulación hasta entrevista o cierre según sector y modalidad?",
        "Entidades_YEB": "applications.created_at, jobs.work_mode, companies.industry",
        "Indicador_Objetivo": "tiempo_respuesta_dias (media y mediana)",
    },
    {
        "ID": "PA-03",
        "Eje_Analitico": "Tasa de postulaciones por vacante",
        "Pregunta_Negocio": "¿Cuál es el volumen promedio de postulantes por vacante publicada según experiencia requerida?",
        "Entidades_YEB": "jobs.id, jobs.status, jobs.experience, applications.id",
        "Indicador_Objetivo": "Postulaciones / Vacante activa",
    },
    {
        "ID": "PA-04",
        "Eje_Analitico": "Brecha por ciudad y salario",
        "Pregunta_Negocio": "¿Qué brecha existe entre los salarios/ciudades ofertados por las empresas y las expectativas/ubicación de los jóvenes?",
        "Entidades_YEB": "jobs.city, jobs.salary_min, jobs.salary_max, users.location",
        "Indicador_Objetivo": "Ratio Candidatos/Vacantes y brecha_salarial_aspiracion (COP)",
    },
]

df_questions = pd.DataFrame(analytical_matrix)
display(df_questions)

,ID,Eje_Analitico,Pregunta_Negocio,Entidades_YEB,Indicador_Objetivo
0,PA-01,Embudo de selección,¿Cuál es la tasa de conversión y descarte entr...,"applications.status, users.is_minor",Tasa de conversión por etapa (%)
1,PA-02,Tiempo promedio de contratación,¿Cuántos días transcurren en promedio desde la...,"applications.created_at, jobs.work_mode, compa...",tiempo_respuesta_dias (media y mediana)
2,PA-03,Tasa de postulaciones por vacante,¿Cuál es el volumen promedio de postulantes po...,"jobs.id, jobs.status, jobs.experience, applica...",Postulaciones / Vacante activa
3,PA-04,Brecha por ciudad y salario,¿Qué brecha existe entre los salarios/ciudades...,"jobs.city, jobs.salary_min, jobs.salary_max, u...",Ratio Candidatos/Vacantes y brecha_salarial_as...


## 4. Declaración Ética: Minimización y Anonimización de Datos Sensibles (`CA-ISS-01-04`)
En cumplimiento de la **Ley 1581 de 2012 (Habeas Data)** y la **Ley 1098 de 2006 (Código de la Infancia y la Adolescencia)**, y para mitigar el riesgo **`LEG-001`**, se aplican tres controles obligatorios sobre los datos de candidatos (especialmente adolescentes de **16 y 17 años**):

1. **Minimización en origen**: Se excluyen de cualquier extracción hacia `data/raw/` las contraseñas (`password`), archivos binarios (`LONGBLOB` de hojas de vida `cv_data`, fotos `profile_photo_data` y soportes de menores `user_minor_documents`), direcciones IP y mensajes privados.
2. **Seudonimización e Identificadores Sintéticos**: Los campos de identificación directa (`name`, `email`, `cedula`, `phone`) se transforman en un hash irreversible (`candidate_hash`) o se sustituyen por datos sintéticos reproducibles con `Faker` (`Faker.seed(42)`).
3. **Salvaguarda de Menores (16–17 años)**: La fecha de nacimiento exacta se generaliza en edad entera (`16`–`24`) y cohorte (`Menor Protegido (16-17)` vs. `Joven Adulto (18-24)`), verificando siempre el estado de autorización legal (`minor_authorized`).

In [4]:
# Demostración práctica y auditable de anonimización sobre una muestra sintética
fake = Faker("es_CO")
Faker.seed(42)
np.random.seed(42)

raw_candidates_sample = [
    {"user_id": 101, "name": "Camila Gómez", "cedula": "1098111222", "age": 17, "is_minor": 1, "minor_blocked": 0, "city": "Medellín", "expected_salary": 1450000},
    {"user_id": 102, "name": "Mateo Ríos", "cedula": "1098333444", "age": 16, "is_minor": 1, "minor_blocked": 1, "city": "Bogotá", "expected_salary": 1350000},
    {"user_id": 103, "name": "Valentina Díaz", "cedula": "1010555666", "age": 21, "is_minor": 0, "minor_blocked": 0, "city": "Cali", "expected_salary": 1800000},
]

def anonymize_candidate_record(record: dict, salt: str = "yeb_analytics_salt_2026") -> dict:
    """Aplica minimización y seudonimización eliminando PII directa."""
    digest = hashlib.sha256(f"{record['user_id']}:{salt}".encode("utf-8")).hexdigest()[:12]
    is_minor_flag = bool(record["is_minor"])
    return {
        "candidate_hash": f"CAND-{digest}",
        "doc_synthetic": f"DOC-SYN-{fake.unique.random_number(digits=6, fix_len=True)}",
        "age": int(record["age"]),
        "age_cohort": "Menor Protegido (16-17)" if is_minor_flag else "Joven Adulto (18-24)",
        "is_minor": is_minor_flag,
        "minor_authorized": (not is_minor_flag) or (record["minor_blocked"] == 0),
        "city": record["city"],
        "expected_salary_cop": float(record["expected_salary"]),
    }

df_anonymized = pd.DataFrame([anonymize_candidate_record(r) for r in raw_candidates_sample])

# Verificaciones de invariantes éticos (CP-041C)
pii_columns = {"name", "cedula", "email", "phone", "birth_date"}
assert pii_columns.isdisjoint(df_anonymized.columns), "Fallo ético: columnas PII presentes en el DataFrame."
assert df_anonymized["age"].between(16, 24).all(), "Fallo de dominio: edades fuera del rango juvenil 16-24."

display(df_anonymized)

,candidate_hash,doc_synthetic,age,age_cohort,is_minor,minor_authorized,city,expected_salary_cop
0,CAND-e34210f9f520,DOC-SYN-770487,17,Menor Protegido (16-17),True,True,Medellín,1450000.0
1,CAND-c8248a8bc34a,DOC-SYN-216739,16,Menor Protegido (16-17),True,False,Bogotá,1350000.0
2,CAND-8e2a5b0ddb0d,DOC-SYN-126225,21,Joven Adulto (18-24),False,True,Cali,1800000.0
